In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DeltaLakeDemo")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark + Delta Lake iniciado!")

26/05/05 20:54:44 WARN Utils: Your hostname, DESKTOP-9CLKLQP resolves to a loopback address: 127.0.1.1; using 172.19.75.57 instead (on interface eth0)
26/05/05 20:54:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/rafael/.cache/pypoetry/virtualenvs/spark-deltalake-iceberg-IhLNl8Fq-py3.11/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rafael/.ivy2/cache
The jars for the packages stored in: /home/rafael/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-355706ff-0251-42e9-985a-2033afea0479;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (466ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (75ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (86ms)
:: resolution report :: resolve -11298ms ::

Spark + Delta Lake iniciado!


In [2]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS clientes (
    id_cliente INT,
    nome STRING,
    email STRING,
    cidade STRING
  ) USING DELTA LOCATION '/tmp/delta/clientes'
""")

spark.sql("""
  CREATE TABLE IF NOT EXISTS produtos (
    id_produto INT,
    nome STRING,
    categoria STRING,
    preco DOUBLE
  ) USING DELTA LOCATION '/tmp/delta/produtos'
""")

spark.sql("""
  CREATE TABLE IF NOT EXISTS pedidos (
    id_pedido INT,
    id_cliente INT,
    id_produto INT,
    quantidade INT,
    valor_total DOUBLE,
    status STRING,
    data_pedido DATE
  ) USING DELTA LOCATION '/tmp/delta/pedidos'
""")

print("Tabelas criadas com sucesso!")


Tabelas criadas com sucesso!


In [3]:
spark.sql("INSERT INTO clientes VALUES (1,'Ana Lima','ana@email.com','São Paulo')")
spark.sql("INSERT INTO clientes VALUES (2,'João Silva','joao@email.com','Curitiba')")
spark.sql("INSERT INTO produtos VALUES (1,'Notebook','Eletrônicos',3500.00)")
spark.sql("INSERT INTO produtos VALUES (2,'Mouse','Periféricos',150.00)")
spark.sql("INSERT INTO pedidos VALUES (1,1,1,1,3500.00,'pendente','2024-01-10')")
spark.sql("INSERT INTO pedidos VALUES (2,2,2,2,300.00,'enviado','2024-01-11')")

print("Dados inseridos!")
spark.sql("SELECT * FROM pedidos").show()


Dados inseridos!


26/05/05 20:57:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|pendente| 2024-01-10|
|        2|         2|         2|         2|      300.0| enviado| 2024-01-11|
+---------+----------+----------+----------+-----------+--------+-----------+



In [4]:
spark.sql("UPDATE pedidos SET status = 'entregue' WHERE id_pedido = 1")
print("Após UPDATE:")
spark.sql("SELECT * FROM pedidos").show()

Após UPDATE:
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|entregue| 2024-01-10|
|        2|         2|         2|         2|      300.0| enviado| 2024-01-11|
+---------+----------+----------+----------+-----------+--------+-----------+



In [5]:
spark.sql("DELETE FROM pedidos WHERE id_pedido = 2")
print("Após DELETE:")
spark.sql("SELECT * FROM pedidos").show()

Após DELETE:
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|entregue| 2024-01-10|
+---------+----------+----------+----------+-----------+--------+-----------+



In [6]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, '/tmp/delta/pedidos')
print("Histórico de versões:")
dt.history().select("version", "timestamp", "operation").show(truncate=False)

print("Estado inicial (versão 0):")
spark.read.format("delta").option("versionAsOf", 0).load("/tmp/delta/pedidos").show()

Histórico de versões:
+-------+-----------------------+------------+
|version|timestamp              |operation   |
+-------+-----------------------+------------+
|4      |2026-05-05 20:57:46.78 |DELETE      |
|3      |2026-05-05 20:57:25.037|UPDATE      |
|2      |2026-05-05 20:57:04.259|WRITE       |
|1      |2026-05-05 20:57:03.921|WRITE       |
|0      |2026-05-05 20:56:40.903|CREATE TABLE|
+-------+-----------------------+------------+

Estado inicial (versão 0):
+---------+----------+----------+----------+-----------+------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|status|data_pedido|
+---------+----------+----------+----------+-----------+------+-----------+
+---------+----------+----------+----------+-----------+------+-----------+

